# Prepare Figure 6 group-level result tables

Run this notebook after `F6_add_calendar_day.ipynb`. It reads the lifetime diagnostic summaries from `A01_cell_lifetime_diagnostics`, assigns each cell to the cycling and calendar-aging groups used in Figure 6, and writes the tables consumed by the Figure 6 plotting scripts.

Outputs are written to `F6_group_plots_outputs_html`:
- `Master_with_groups_and_day.csv`
- `CycleMaster.csv`
- `CalendarMaster.csv`
- lightweight HTML previews for each group


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import plotly.express as px
    HAVE_PLOTLY = True
except Exception:
    HAVE_PLOTLY = False

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)


## 1. Paths and group definitions


In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "code" / "diagnostic_algorithm_lifetime_crate").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repository root.")

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "F6_new_framework_group_results_python.ipynb").exists():
    NOTEBOOK_DIR = Path("code/plotting/f6").resolve()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)

BATCH_RESULTS_ROOT = REPO_ROOT / "code" / "diagnostic_algorithm_lifetime_crate" / "batch_results" / "A01_cell_lifetime_diagnostics"
CSV_GLOB = "cell*/cell*_voltage_fit_summary.csv"
OUT_DIR = NOTEBOOK_DIR / "F6_group_plots_outputs_html"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CYCLE_GROUPS = {
    "Pressure_25psi_25C": [1, 35, 60],
    "Pressure_5psi_25C": [2, 7, 47],
    "Temp_45C_15psi": [17, 25, 72],
    "Temp_0C_15psi": [21, 58, 68],
    "P25_T45": [27, 71],
    "P25_T0": [16, 32, 49],
    "Baseline_15psi_25C": [31, 44, 65],
    "0p6C1D": [18, 23, 53],
    "0p6C_0p6D": [30, 51, 61],
    "1C1D": [8, 24, 66],
    "2C2D": [56, 74],
    "DCFC": [28, 41, 43],
    "DOD_5_96": [9, 40, 54],
    "DOD_20_80": [22, 26, 75],
    "DOD_50_100": [11, 19, 55],
    "DOD_0_50": [20],
    "Multi_rate": [59],
}

CALENDAR_GROUPS = {
    "Cal_100SOC_25C": [5, 46],
    "Cal_100SOC_45C": [36, 52],
    "Cal_100SOC_60C": [3],
    "Cal_80SOC_45C": [4, 57],
    "Cal_80SOC_60C": [42, 48],
    "Cal_50SOC_45C": [45],
}

print("REPO_ROOT =", REPO_ROOT)
print("BATCH_RESULTS_ROOT =", BATCH_RESULTS_ROOT)
print("OUT_DIR =", OUT_DIR)


## 2. Load lifetime summary CSVs


In [ ]:
def infer_cell_from_path(path_str: str):
    match = re.search(r"cell(\d{3})", str(path_str), flags=re.IGNORECASE)
    return int(match.group(1)) if match else np.nan

csv_paths = sorted(BATCH_RESULTS_ROOT.glob(CSV_GLOB))
print(f"Found {len(csv_paths)} lifetime summary CSV files")

dfs = []
for summary_path in csv_paths:
    df = pd.read_csv(summary_path)
    cell = infer_cell_from_path(str(summary_path))
    if "cell" not in df.columns or df["cell"].isna().all():
        df["cell"] = cell
    try:
        df["source_file"] = str(summary_path.relative_to(REPO_ROOT))
    except ValueError:
        df["source_file"] = summary_path.name
    dfs.append(df)

Master = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print("Master shape:", Master.shape)
Master.head()


## 3. Standardize plotting aliases


In [ ]:
for col in [
    "cell", "rpt_seq", "rpt_key", "Ah_throughput", "Qcc_max_meas", "Cn_Si", "Cn_Gr", "Cn", "Cp", "LLI",
    "x100", "y100", "x0", "y0", "x100_si", "x100_gr", "x0_si", "x0_gr", "si_scale_a", "si_shift_b",
    "rmse_v_global", "rmse_dvdq_global", "C_est", "day", "transition_soc",
]:
    if col in Master.columns:
        Master[col] = pd.to_numeric(Master[col], errors="coerce")

if "Cn" not in Master.columns and {"Cn_Si", "Cn_Gr"}.issubset(Master.columns):
    Master["Cn"] = Master["Cn_Si"] + Master["Cn_Gr"]

Master["AhTh"] = Master["Ah_throughput"] if "Ah_throughput" in Master.columns else np.nan
Master["C"] = Master["Qcc_max_meas"] if "Qcc_max_meas" in Master.columns else np.nan
Master["lli"] = Master["LLI"] if "LLI" in Master.columns else np.nan
Master["CnSi"] = Master["Cn_Si"] if "Cn_Si" in Master.columns else np.nan
Master["CnGr"] = Master["Cn_Gr"] if "Cn_Gr" in Master.columns else np.nan
Master["RMSE_V"] = Master["rmse_v_global"] * 1000 if "rmse_v_global" in Master.columns else np.nan
Master["RMSE_dVdQ"] = Master["rmse_dvdq_global"] if "rmse_dvdq_global" in Master.columns else np.nan

sort_cols = [col for col in ["cell", "AhTh", "rpt_seq"] if col in Master.columns]
Master = Master.sort_values(sort_cols, na_position="last").reset_index(drop=True)
Master.head()


## 4. Assign Figure 6 groups


In [ ]:
def invert_group_map(group_map):
    out = {}
    for group_name, cells in group_map.items():
        for cell in cells:
            out[int(cell)] = group_name
    return out

cycle_lookup = invert_group_map(CYCLE_GROUPS)
calendar_lookup = invert_group_map(CALENDAR_GROUPS)

Master["cycle_group"] = Master["cell"].map(cycle_lookup)
Master["calendar_group"] = Master["cell"].map(calendar_lookup)
Master["aging_type"] = np.select(
    [Master["calendar_group"].notna(), Master["cycle_group"].notna()],
    ["calendar", "cycle"],
    default="unassigned",
)

# Remove clearly invalid capacity estimates before preparing Figure 6 tables.
# These rows come from non-target or failed calendar RPT fits and otherwise
# create artificial drops in the calendar-aging trends.
cap_col = "Qcc_max_meas"
if cap_col in Master.columns:
    before = len(Master)
    Master = Master.loc[Master[cap_col].notna() & (Master[cap_col] >= 1.0)].copy()
    print(f"Removed {before - len(Master)} rows with {cap_col} < 1 Ah")

# Calendar-aging summaries can contain duplicate rows at the same calendar
# day for a given cell. Keep the physically meaningful CC-capacity estimate.
if "day" in Master.columns and cap_col in Master.columns:
    is_calendar_row = Master["calendar_group"].notna()
    Master_cycle_part = Master.loc[~is_calendar_row].copy()
    Master_calendar_part = Master.loc[is_calendar_row].copy()

    if not Master_calendar_part.empty:
        before_calendar = len(Master_calendar_part)
        Master_calendar_part["_day_key"] = Master_calendar_part["day"].round(6)
        Master_calendar_part = (
            Master_calendar_part
            .sort_values(["cell", "_day_key", cap_col], ascending=[True, True, False])
            .drop_duplicates(subset=["cell", "_day_key"], keep="first")
            .drop(columns=["_day_key"])
        )
        print(f"Removed {before_calendar - len(Master_calendar_part)} duplicate calendar rows")

    Master = pd.concat([Master_cycle_part, Master_calendar_part], ignore_index=True)
    Master = Master.sort_values(sort_cols, na_position="last").reset_index(drop=True)

CycleMaster = Master.loc[Master["cycle_group"].notna()].copy()
CalendarMaster = Master.loc[Master["calendar_group"].notna()].copy()

print("Cycle rows:", CycleMaster.shape)
print("Calendar rows:", CalendarMaster.shape)
print("Unassigned cells:", sorted(Master.loc[Master["aging_type"] == "unassigned", "cell"].dropna().astype(int).unique().tolist()))
print("Cycle group counts:")
print(CycleMaster.groupby("cycle_group")["cell"].nunique().sort_index())
print("Calendar group counts:")
print(CalendarMaster.groupby("calendar_group")["cell"].nunique().sort_index())


## 5. Save CSV outputs


In [ ]:
Master.to_csv(OUT_DIR / "Master_with_groups_and_day.csv", index=False)
CycleMaster.to_csv(OUT_DIR / "CycleMaster.csv", index=False)
CalendarMaster.to_csv(OUT_DIR / "CalendarMaster.csv", index=False)
print("Saved grouped CSV outputs to", OUT_DIR)


## 6. Generate lightweight HTML previews


In [ ]:
def write_group_html(df, group_col, x_col, out_subdir, title_prefix):
    out_subdir = OUT_DIR / out_subdir
    out_subdir.mkdir(parents=True, exist_ok=True)
    metrics = [col for col in ["C", "CnSi", "LLI", "transition_soc"] if col in df.columns]
    if "transition_soc" in metrics and df["transition_soc"].dropna().max() <= 1.5:
        plot_df = df.copy()
        plot_df["transition_soc"] = plot_df["transition_soc"] * 100
    else:
        plot_df = df.copy()

    for group_name, group_df in plot_df.dropna(subset=[group_col]).groupby(group_col):
        group_df = group_df.sort_values(["cell", x_col, "rpt_seq"], na_position="last")
        html_path = out_subdir / f"{group_name}.html"
        if HAVE_PLOTLY and metrics and x_col in group_df.columns:
            long_df = group_df[["cell", x_col, *metrics]].melt(id_vars=["cell", x_col], value_vars=metrics, var_name="metric", value_name="value")
            fig = px.line(
                long_df.dropna(subset=[x_col, "value"]),
                x=x_col,
                y="value",
                color="cell",
                facet_row="metric",
                markers=True,
                title=f"{title_prefix}: {group_name}",
                height=260 * max(1, len(metrics)),
            )
            fig.update_yaxes(matches=None, showticklabels=True)
            fig.write_html(html_path, include_plotlyjs="cdn")
        else:
            group_df.to_html(html_path, index=False)

write_group_html(CycleMaster, "cycle_group", "AhTh", "cycle_group_plots_html", "Cycle group")
write_group_html(CalendarMaster, "calendar_group", "day", "calendar_group_plots_html", "Calendar group")

compare_dir = OUT_DIR / "compare_group_plots_html"
compare_dir.mkdir(parents=True, exist_ok=True)
high_temp_groups = ["Pressure_25psi_25C", "P25_T45", "P25_T0"]
high_temp = CycleMaster.loc[CycleMaster["cycle_group"].isin(high_temp_groups)].copy()
if HAVE_PLOTLY and not high_temp.empty:
    plot_df = high_temp.copy()
    if "transition_soc" in plot_df.columns and plot_df["transition_soc"].dropna().max() <= 1.5:
        plot_df["transition_soc"] = plot_df["transition_soc"] * 100
    long_df = plot_df[["cycle_group", "cell", "AhTh", "C", "CnSi", "LLI", "transition_soc"]].melt(
        id_vars=["cycle_group", "cell", "AhTh"],
        value_vars=[col for col in ["C", "CnSi", "LLI", "transition_soc"] if col in plot_df.columns],
        var_name="metric",
        value_name="value",
    )
    fig = px.line(long_df.dropna(subset=["AhTh", "value"]), x="AhTh", y="value", color="cycle_group", line_group="cell", facet_row="metric", markers=True, title="Temperature comparison")
    fig.update_yaxes(matches=None, showticklabels=True)
    fig.write_html(compare_dir / "HighTemp_comparison.html", include_plotlyjs="cdn")
elif not high_temp.empty:
    high_temp.to_html(compare_dir / "HighTemp_comparison.html", index=False)

print("HTML previews written to", OUT_DIR)


## 7. Quick summary


In [ ]:
# Summary columns retain saved-output aliases si_scale_a and si_shift_b for
# the manuscript parameters s_V and U_off.
summary_cols = [col for col in ["C", "CnSi", "CnGr", "Cn", "Cp", "LLI", "si_scale_a", "si_shift_b", "transition_soc"] if col in Master.columns]
summary = Master.groupby("aging_type")[summary_cols].agg(["count", "mean", "std", "median"])
summary
